In [ ]:
%%capture
%pip install gradio==4.44.0
%pip install --upgrade gradio
%pip install langchain==0.2.11
%pip install langchain-community==0.2.10
%pip install chromadb==0.4.24
%pip install --upgrade chromadb
%pip install jq
%pip install --force-reinstall numpy==1.26.4


In [ ]:
####### Start Ollama server #########

!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start the server in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to initialize
time.sleep(5)
print("Ollama server is running in the background!")
!ollama pull llama3

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import JSONLoader
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
import gradio as gr
import logging
import sys
import warnings


# To suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')


## Document loader
def document_loader(file):
    if file is None:
        logging.info( "Please upload a valid JSON file.")
    loader = JSONLoader(
        file_path=file,
        jq_schema='.[]', # Adjust the jq_schema based on your JSON structure
        text_content=False
    )
    loaded_document = loader.load()
    return loaded_document

## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks


## Token embedding
def hf_embedding():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': False}

    embedding_model = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embedding_model

## Initialize vector db
def vector_database(chunks):
    embedding_model = hf_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb


#Create vector db
def create_vector_db(file):
    if file is None:
        logging.info("Please upload a valid JSON file.")
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever()


## Query Chain
def retriever_qa(retriever_state, query):

    if retriever_state is None:
        logging.error("The provided retriever object is None.")
        return "Error: No retriever available to search the context."


    # 1. Define custom prompt template
    template = """You are an expert fact-extraction system. You must output EXACTLY the correct answer from the provided context. Do NOT use conversational filler, do NOT add introductory text like 'According to the context', and do NOT output full sentences. Only output the exact answer.
    Context: {context}
    Question: {question}
    Answer:"""

    QA_CHAIN_PROMPT = PromptTemplate(
        input_variables=["context", "question"],
        template=template,
    )

    llm = Ollama(model="llama3")

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever_state,
        return_source_documents=True,
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
    )
    try:
        response = qa.invoke(query)
        if isinstance(response, dict):
            answer = response.get("result", "").strip()
            docs = response.get("source_documents", [])
        else:
            answer = str(response).strip()
            docs = retriever_state.get_relevant_documents(query)
        context = "\n".join([doc.page_content for doc in docs])

    except Exception as e:
        logging.error(f"Error during chain invocation: {e}")
        answer = "The LLM could not generate an answer from the retrieved context."
    return answer



# Gradio Interface using State
with gr.Blocks() as rag_application:
    gr.Markdown("# AI: Upload a JSON document and ask any question.")

    retriever_state = gr.State() # Store the retriever object in memory


    with gr.Row():
        file_input = gr.File(label="Upload JSON File", file_count="single", file_types=['.json'], type="filepath")
        process_button = gr.Button("Process JSON")

    with gr.Row():
        query_input = gr.Textbox(label="Input Query", lines=2, placeholder="Type your question here...")
        output_box = gr.Textbox(label="Output")

    with gr.Row():
        submit_query_btn = gr.Button("Submit Question")
    process_button.click(fn=create_vector_db, inputs=file_input, outputs=retriever_state)
    submit_query_btn.click(fn=retriever_qa, inputs=[retriever_state, query_input], outputs=output_box)


# Launch the app
rag_application.launch(server_name="127.0.0.1",share=True)


In [ ]:
rag_application.close()